In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("devicharith/language-translation-englishfrench")

print("Path to dataset files:", path)

import os

for root, dirs, files in os.walk(path):
    for file in files:
        l = os.path.join(root, file)


import pandas as pd
file_path = os.path.join(path, l)
df = pd.read_csv(file_path)
df.head()
df = df.sample(
    n=20000,
    random_state=42
).reset_index(drop=True)

100%|██████████| 3.51M/3.51M [00:00<00:00, 166MB/s]

Extracting files...
Path to dataset files: /root/.cache/kagglehub/datasets/devicharith/language-translation-englishfrench/versions/2


In [ ]:
import re
def tokenize(sent):
  sent = sent.lower()
  sent = re.sub(r"([.,!?;:])", r" \1", sent)
  return sent.split()

from collections import Counter
#Counter(df["English words/sentences"])

class Vocab:
  def __init__(self):
    self.word2idx = {
        "<PAD>" : 0,
        "<UNK>" : 1,
        "<SOS>" : 2,
        "<EOS>" : 3
    }

    self.idx2word = {
        0 : "<PAD>",
        1 : "<UNK>",
        2 : "<SOS>",
        3: "<EOS>",

    }
  def build(self, sents, min_freq = 1):
    counter = Counter()
    for sent in sents.tolist():
      t = tokenize(sent)
      counter.update(t)

    for word, freq in counter.items():
      if freq >= min_freq:
        idx = len(self.word2idx)
        self.word2idx[word] = idx
        self.idx2word[idx] = word

    def numericalize(self, sent):
      tokens = tokenize(sent)
      return [self.word2idx.get(token, self.word2idx["<UNK>"]) for token in tokens]

    def __len__(self):
      return len(self.word2idx)


from collections import Counter
#Counter(df["English words/sentences"])

class Vocab:
  def __init__(self):
    self.word2idx = {
        "<PAD>" : 0,
        "<UNK>" : 1,
        "<SOS>" : 2,
        "<EOS>" : 3
    }

    self.idx2word = {
        0 : "<PAD>",
        1 : "<UNK>",
        2 : "<SOS>",
        3: "<EOS>",
    }

  def build(self, sents, min_freq = 1):
    counter = Counter()
    for sent in sents.tolist():
      t = tokenize(sent)
      counter.update(t)

    for word, freq in counter.items():
      if freq >= min_freq:
        idx = len(self.word2idx)
        self.word2idx[word] = idx
        self.idx2word[idx] = word

  def numericalize(self, sent):
      tokens = tokenize(sent)
      return ( [self.word2idx["<SOS>"]] + [self.word2idx.get(token, self.word2idx["<UNK>"]) for token in tokens] + [self.word2idx["<EOS>"]])
  def __len__(self):
      return len(self.word2idx)

english_vocab = Vocab()
french_vocab = Vocab()
english_vocab.build(df["English words/sentences"])
french_vocab.build(df["French words/sentences"])

print("\nEnglish:")
print(list(english_vocab.word2idx.items())[:20])
print("\nFrench:")
print(list(french_vocab.word2idx.items())[:20])


from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(df, test_size= 0.2, random_state = 42)

import torch
from torch.utils.data import Dataset

class TranslateDataset(Dataset):
  def __init__(self, df, english_vocab, french_vocab):
    self.data = df.reset_index(drop = True)

    self.eng = english_vocab
    self.fre = french_vocab

  def __len__(self):
    return len(self.data)

  def __getitem__(self, index):
    en = self.data.iloc[index]["English words/sentences"]
    fr = self.data.iloc[index]["French words/sentences"]

    engl_ids = self.eng.numericalize(en)
    fren_ids = self.fre.numericalize(fr)

    return (
        torch.tensor(engl_ids, dtype = torch.long),
        torch.tensor(fren_ids, dtype = torch.long)
    )

train_dataset = TranslateDataset(
    train_df,
    english_vocab,
    french_vocab
)

val_dataset = TranslateDataset(
    val_df,
    english_vocab,
    french_vocab
)

print("Training dataset:", len(train_dataset))
print("Validation dataset:", len(val_dataset))

from torch.nn.utils.rnn import pad_sequence

def padding(batch):
  input = []
  label = []

  for i, l in batch:
    input.append(i)
    label.append(l)

  input = pad_sequence( input, batch_first= True, padding_value= english_vocab.word2idx["<PAD>"])
  label = pad_sequence( label, batch_first= True, padding_value= french_vocab.word2idx["<PAD>"])

  return input, label


from torch.utils.data import DataLoader

BATCH_SIZE = 32

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn= padding
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn= padding
)

import re
from collections import Counter


English:
[('<PAD>', 0), ('<UNK>', 1), ('<SOS>', 2), ('<EOS>', 3), ('take', 4), ('a', 5), ('seat', 6), ('.', 7), ('i', 8), ('wish', 9), ('tom', 10), ('was', 11), ('here', 12), ('how', 13), ('did', 14), ('the', 15), ('audition', 16), ('go', 17), ('?', 18), ("i've", 19)]

French:
[('<PAD>', 0), ('<UNK>', 1), ('<SOS>', 2), ('<EOS>', 3), ('prends', 4), ('place', 5), ('!', 6), ("j'aimerais", 7), ('que', 8), ('tom', 9), ('soit', 10), ('là', 11), ('.', 12), ('comment', 13), ("s'est", 14), ('passée', 15), ("l'audition", 16), ('?', 17), ('je', 18), ("n'ai", 19)]
Training dataset: 16000
Validation dataset: 4000


In [ ]:
dim_model = 128 #128 positional values ie. 128 embedding dimentions
lem_max = 100 #100 positions

import math
import torch
import torch.nn as nn

class PositionalEncoding(nn.Module):
  def __init__(self, dim_model, len_max= 100):
    super().__init__()
    pos_en = torch.zeros(len_max, dim_model)
    position = torch.arange(0, len_max, dtype = torch.float).unsqueeze(1)
    div_term = torch.exp(torch.arange(0, dim_model, 2).float() * (- math.log(10000.0)/dim_model))
    pos_en[:, 0::2] = torch.sin(position * div_term) #every 2nd dim starting from 0th
    pos_en[:, 1::2] = torch.cos(position * div_term) #every 2nd dim starting from 1st
    pos_en = pos_en.unsqueeze(0)
    self.register_buffer(
        "pos_en", pos_en  #not trainable
    )
  def forward(self, x):
    x = x + self.pos_en[:,:x.size(1), :]
    return x


In [ ]:
pos_encoder = PositionalEncoding( dim_model=128, len_max=100)
x = torch.zeros(2, 10, 128)
output = pos_encoder(x)

print(output.shape)

torch.Size([2, 10, 128])


In [ ]:
"""class ScaledDotProductAttention(nn.Module):

    def __init__(self, dropout=0.0):
        super().__init__()

        self.dropout = nn.Dropout(dropout)

    def forward(self, Q, K, V, mask=None):

        d_k = Q.size(-1)

        scores = torch.matmul(
            Q,
            K.transpose(-2, -1)
        ) / math.sqrt(d_k)

        if mask is not None:
            scores = scores.masked_fill(
                mask == 0,
                -1e9
            )

        attention_weights = torch.softmax(
            scores,
            dim=-1
        )

        attention_weights = self.dropout(
            attention_weights
        )

        output = torch.matmul(
            attention_weights,
            V
        )

        return output, attention_weights"""

In [ ]:
#Multihead Attention

In [ ]:
class FeedForward(nn.Module):
  def __init__(self, dim_model, pf_dim, dropout = 0.0):
    super().__init__()
    self.fc1 = nn.Linear(dim_model, pf_dim)
    self.fc2 = nn.Linear(pf_dim, dim_model)
    self.dropout = nn.Dropout(dropout)


  def forward(self, x):
    x = self.fc1(x)
    x = torch.relu(x)
    x = self.dropout(x)
    x = self.fc2(x)
    return x



In [ ]:
class EncoderLayer(nn.Module):
  def __init__(self, dim_model, n_heads, pf_dim, dropout = 0.0):
    super().__init__()
    self.attention = nn.MultiheadAttention(embed_dim= dim_model, num_heads= n_heads, dropout= dropout, batch_first = True)
    self.feed_forward = FeedForward(dim_model, pf_dim, dropout)
    self.norm1 = nn.LayerNorm(dim_model)
    self.norm2 = nn.LayerNorm(dim_model)
    self.norm3 = nn.LayerNorm(dim_model)
    self.dropout = nn.Dropout(dropout)

  def forward(self, src, src_padding_mask = None):

    attentioon_op, attention_wgts = self.attention(src, src, src, key_padding_mask = src_padding_mask)
    src = self.norm1(src + self.dropout(attentioon_op))
    ff_op = self.feed_forward(src)
    src = self.norm2(
        src + self.dropout(ff_op)

    )
    return src, attention_wgts

In [ ]:
D_MODEL = 256
N_HEADS = 8
PF_DIM = 512
DROPOUT = 0.1
encoder_layer = EncoderLayer( dim_model=D_MODEL, n_heads=N_HEADS, pf_dim=PF_DIM, dropout=DROPOUT)

batch_size = 32
src_len = 20
x = torch.randn( batch_size, src_len, D_MODEL)

output, attention_weights = encoder_layer(x)
print("Input shape :", x.shape)
print("Output shape:", output.shape)
print("Attention shape:", attention_weights.shape)

Input shape : torch.Size([32, 20, 256])
Output shape: torch.Size([32, 20, 256])
Attention shape: torch.Size([32, 20, 20])


In [ ]:
class DecoderLayer(nn.Module):
  def __init__(self, dim_model, n_heads, pf_dim, dropout = 0.0):
    super().__init__()
    self.self_attention = nn.MultiheadAttention( embed_dim= dim_model, num_heads= n_heads, dropout= dropout, batch_first= True)
    self.cross_attention =nn.MultiheadAttention( embed_dim= dim_model, num_heads= n_heads, dropout= dropout, batch_first= True)
    self.feed_forward = FeedForward(dim_model, pf_dim, dropout)
    self.norm1 = nn.LayerNorm(dim_model)
    self.norm2 = nn.LayerNorm(dim_model)
    self.norm3 = nn.LayerNorm(dim_model)
    self.dropout = nn.Dropout(dropout)

  def forward(self, trg, encoder_output, trg_mask = None, trg_padding_mask=None, src_padding_mask = None):
    self_attention_op, self_attention_wgts = self.self_attention(trg, trg, trg, attn_mask = trg_mask, key_padding_mask = trg_padding_mask)
    trg = self.norm1(trg + self.dropout(self_attention_op))
    cross_attention_op, cross_attention_wgts = self.cross_attention(trg, encoder_output, encoder_output, key_padding_mask = src_padding_mask)
    trg = self.norm2(trg + self.dropout(cross_attention_op))
    ff_op = self.feed_forward(trg)
    trg = self.norm3(
        trg + self.dropout(ff_op)

    )
    return trg, self_attention_wgts, cross_attention_wgts

In [ ]:
D_MODEL = 256
N_HEADS = 8
PF_DIM = 512
DROPOUT = 0.1

decoder_layer = DecoderLayer( dim_model=D_MODEL, n_heads=N_HEADS, pf_dim=PF_DIM, dropout=DROPOUT )
batch_size = 32
trg_len = 15
src_len = 20
trg = torch.randn( batch_size, trg_len, D_MODEL)

encoder_output = torch.randn( batch_size, src_len, D_MODEL)
output, self_weights, cross_weights = decoder_layer( trg, encoder_output)

print("Target input :", trg.shape)
print("Encoder output:", encoder_output.shape)
print("Output        :", output.shape)
print("Self attention:", self_weights.shape)
print("Cross attention:", cross_weights.shape)

Target input : torch.Size([32, 15, 256])
Encoder output: torch.Size([32, 20, 256])
Output        : torch.Size([32, 15, 256])
Self attention: torch.Size([32, 15, 15])
Cross attention: torch.Size([32, 15, 20])


In [ ]:
class TransformerEncoder(nn.Module):
  def __init__(self, input_dim, d_model, n_heads, pf_dim, n_layers, dropout = 0.0):
    super().__init__()
    self.embedding = nn.Embedding(input_dim, d_model, padding_idx= english_vocab.word2idx["<PAD>"])
    self.positional_encoding = PositionalEncoding(
        d_model
    )
    self.layers = nn.ModuleList([
            EncoderLayer( d_model, n_heads, pf_dim, dropout)
            for _ in range(n_layers) ])
    self.dropout = nn.Dropout(dropout)
    self.d_model = d_model

  def forward(self, src, src_padding_mask = None):
    src = self.embedding(src)
    src = src * math.sqrt(self.d_model)
    src = self.positional_encoding(src)
    src = self.dropout(src)
    attention_weights = []
    for layer in self.layers:

            src, attention = layer( src, src_padding_mask)
            attention_weights.append(attention)
    return src, attention_weights

In [ ]:
n_layers = 3
class TransformerDecoder(nn.Module):

    def __init__( self, output_dim, d_model, n_heads, pf_dim,  n_layers, dropout=0.0 ):
        super().__init__()

        self.embedding = nn.Embedding( output_dim, d_model, padding_idx=french_vocab.word2idx["<PAD>"])

        self.positional_encoding = PositionalEncoding( d_model)

        self.layers = nn.ModuleList([
            DecoderLayer( d_model, n_heads, pf_dim, dropout)
            for _ in range(n_layers) ])

        self.fc_out = nn.Linear( d_model, output_dim)
        self.dropout = nn.Dropout(dropout)

        self.d_model = d_model


    def forward( self,trg, encoder_output, trg_mask=None,trg_padding_mask=None,src_padding_mask=None):

        trg = self.embedding(trg)
        trg = trg * math.sqrt(self.d_model)
        trg = self.positional_encoding(trg)
        trg = self.dropout(trg)
        self_attention_weights = []
        cross_attention_weights = []
        for layer in self.layers:
          trg, self_attention, cross_attention = layer(trg,encoder_output,trg_mask, trg_padding_mask,src_padding_mask)
          self_attention_weights.append(self_attention)

          cross_attention_weights.append( cross_attention )

        output = self.fc_out(trg)

        return output,self_attention_weights,cross_attention_weights




In [ ]:
class Transformer(nn.Module):
  def __init__(self, encoder, decoder, device):
    super().__init__()

    self.encoder = encoder
    self.decoder = decoder
    self.device = device

  def forward(self, src, trg):
    src_padding_mask = create_src_mask(src)
    trg_padding_mask = create_trg_padding_mask(trg)
    trg_mask = create_trg_mask(trg)
    encoder_output, encoder_attention = self.encoder( src,src_padding_mask)
    output, decoder_self_attention, decoder_cross_attention = self.decoder(trg, encoder_output, trg_mask ,trg_padding_mask, src_padding_mask)

    return output,encoder_attention, decoder_self_attention, decoder_cross_attention


In [ ]:
input_dim = len(english_vocab)
output_dim = len(french_vocab)

d_model = 256
n_heads = 8
pf_dim= 512
n_layers = 3
dropout = 0.1

encoder = TransformerEncoder(output_dim, d_model, n_heads, pf_dim,  n_layers, dropout)
decoder = TransformerDecoder(output_dim, d_model, n_heads, pf_dim,  n_layers, dropout)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = Transformer( encoder, decoder, device).to(device)

In [ ]:
def create_src_mask(src):
    PAD_IDX = english_vocab.word2idx["<PAD>"]
    src_mask = (src == PAD_IDX)

    return src_mask

def create_trg_padding_mask(trg):
    PAD_IDX = french_vocab.word2idx["<PAD>"]
    return (trg == PAD_IDX)

def create_trg_mask(trg):
    trg_len = trg.shape[1]
    trg_mask = torch.triu(
        torch.ones(trg_len, trg_len, device=trg.device), diagonal=1).bool()
    return trg_mask




In [ ]:
src, trg = next(iter(train_loader))

src = src.to(device)
trg = trg.to(device)

print("src:", src.shape)
print("trg:", trg.shape)

trg_input = trg[:, :-1]
trg_output = trg[:, 1:]

print("trg_input:", trg_input.shape)
print("trg_output:", trg_output.shape)

with torch.no_grad():
    output, encoder_attention, decoder_self_attention, decoder_cross_attention = model(src, trg_input)

print("output:", output.shape)
src_padding_mask = create_src_mask(src)
trg_padding_mask = create_trg_padding_mask(trg_input)
trg_mask = create_trg_mask(trg_input)

print("src padding mask:", src_padding_mask.shape)
print("trg padding mask:", trg_padding_mask.shape)
print("trg causal mask:", trg_mask.shape)

print("encoder attention:", encoder_attention[0].shape)
print("decoder self attention:", decoder_self_attention[0].shape)
print("decoder cross attention:", decoder_cross_attention[0].shape)

src, trg = next(iter(train_loader))
src = src.to(device)
trg = trg.to(device)

trg_input = trg[:, :-1]
trg_output = trg[:, 1:]
print("src:", src.shape)
print("trg:", trg.shape)
print("trg_input:", trg_input.shape)
print("trg_output:", trg_output.shape)

with torch.no_grad():
    output, encoder_attention, decoder_self_attention, decoder_cross_attention = model(src, trg_input)

print("output:", output.shape)
src_padding_mask = create_src_mask(src)
trg_padding_mask = create_trg_padding_mask(trg_input)
trg_mask = create_trg_mask(trg_input)
print("src padding mask:", src_padding_mask.shape)
print("trg padding mask:", trg_padding_mask.shape)
print("trg causal mask:", trg_mask.shape)

print("encoder attention:", encoder_attention[0].shape)
print("decoder self attention:", decoder_self_attention[0].shape)
print("decoder cross attention:", decoder_cross_attention[0].shape)

src: torch.Size([32, 17])
trg: torch.Size([32, 20])
trg_input: torch.Size([32, 19])
trg_output: torch.Size([32, 19])
output: torch.Size([32, 19, 11050])
src padding mask: torch.Size([32, 17])
trg padding mask: torch.Size([32, 19])
trg causal mask: torch.Size([19, 19])
encoder attention: torch.Size([32, 17, 17])
decoder self attention: torch.Size([32, 19, 19])
decoder cross attention: torch.Size([32, 19, 17])
src: torch.Size([32, 13])
trg: torch.Size([32, 16])
trg_input: torch.Size([32, 15])
trg_output: torch.Size([32, 15])
output: torch.Size([32, 15, 11050])
src padding mask: torch.Size([32, 13])
trg padding mask: torch.Size([32, 15])
trg causal mask: torch.Size([15, 15])
encoder attention: torch.Size([32, 13, 13])
decoder self attention: torch.Size([32, 15, 15])
decoder cross attention: torch.Size([32, 15, 13])


In [ ]:
from tqdm import tqdm
def train_transformer(model, iterator, optimizer, criterion, clip=1):

    model.train()

    epoch_loss = 0
    progress = tqdm(iterator, desc="Training")

    for src, trg in iterator:

        src = src.to(device)
        trg = trg.to(device)
        optimizer.zero_grad()
        trg_input = trg[:, :-1]
        trg_output = trg[:, 1:]
        output, _, _, _ = model(src, trg_input)
        output_dim = output.shape[-1]

        output = output.reshape(-1, output_dim)
        trg_output = trg_output.reshape(-1)
        loss = criterion(output, trg_output)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
        optimizer.step()
        epoch_loss += loss.item()

        progress.set_postfix(loss=f"{loss.item():.3f}")
    return epoch_loss / len(iterator)

In [ ]:
def evaluate_transformer(model, iterator, criterion):
    model.eval()
    epoch_loss = 0

    with torch.no_grad():
        for src, trg in iterator:
            src = src.to(device)
            trg = trg.to(device)
            trg_input = trg[:, :-1]
            trg_output = trg[:, 1:]

            output, _, _, _ = model(src, trg_input)
            output_dim = output.shape[-1]
            output = output.reshape(-1, output_dim)
            trg_output = trg_output.reshape(-1)

            loss = criterion(output, trg_output)
            epoch_loss += loss.item()
    return epoch_loss / len(iterator)

In [ ]:
PAD_IDX = french_vocab.word2idx["<PAD>"]

criterion = nn.CrossEntropyLoss(
    ignore_index=PAD_IDX
)
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.0005
)

def count_parameters(model):
    return sum(p.numel()
        for p in model.parameters() if p.requires_grad)

print(f"Trainable parameters: {count_parameters(model):,}")



Trainable parameters: 11,298,346


In [ ]:
import time

N_EPOCHS = 5

loss_train_transformer = []
loss_val_transformer = []

best_valid_loss = float("inf")

for epoch in range(N_EPOCHS):

    start_time = time.time()

    train_loss = train_transformer(
        model,
        train_loader,
        optimizer,
        criterion
    )

    valid_loss = evaluate_transformer(
        model,
        val_loader,
        criterion
    )

    end_time = time.time()

    epoch_mins = int((end_time - start_time) // 60)
    epoch_secs = int((end_time - start_time) % 60)

    loss_train_transformer.append(train_loss)
    loss_val_transformer.append(valid_loss)

    if valid_loss < best_valid_loss:

        best_valid_loss = valid_loss

        torch.save(
            model.state_dict(),
            "translation_model_transformer.pt"
        )

    print(
        f"Epoch: {epoch+1:02} | "
        f"Time: {epoch_mins}m {epoch_secs}s"
    )

    print(
        f"\tTrain Loss: {train_loss:.3f}"
    )

    print(
        f"\tVal. Loss: {valid_loss:.3f}"
    )

Training:   0%|          | 0/500 [06:49<?, ?it/s]


Epoch: 01 | Time: 7m 12s
	Train Loss: 4.415
	Val. Loss: 3.704


Training:   0%|          | 0/500 [06:31<?, ?it/s]


Epoch: 02 | Time: 6m 54s
	Train Loss: 3.300
	Val. Loss: 3.227


Training:   0%|          | 0/500 [06:27<?, ?it/s]


Epoch: 03 | Time: 6m 49s
	Train Loss: 2.704
	Val. Loss: 2.937


Training:   0%|          | 0/500 [06:23<?, ?it/s]


Epoch: 04 | Time: 6m 45s
	Train Loss: 2.244
	Val. Loss: 2.771


Training:   0%|          | 0/500 [06:28<?, ?it/s]


Epoch: 05 | Time: 6m 50s
	Train Loss: 1.867
	Val. Loss: 2.679


In [ ]:
loss_train_transformer


[4.414538484096527,
 3.3000232439041137,
 2.7040608472824097,
 2.2439384257793424,
 1.8672260043621063]

In [ ]:
loss_val_transformer

[3.7044644470214845,
 3.2269804153442383,
 2.9366209354400636,
 2.77073268699646,
 2.6793517513275145]

In [ ]:
model.load_state_dict(
    torch.load(
        "translation_model_transformer.pt",
        map_location=device
    )
)

model.eval()

Transformer(
  (encoder): TransformerEncoder(
    (embedding): Embedding(6541, 256, padding_idx=0)
    (positional_encoding): PositionalEncoding()
    (layers): ModuleList(
      (0-2): 3 x EncoderLayer(
        (attention): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=256, out_features=256, bias=True)
        )
        (feed_forward): FeedForward(
          (fc1): Linear(in_features=256, out_features=512, bias=True)
          (fc2): Linear(in_features=512, out_features=256, bias=True)
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (norm1): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
        (norm3): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
        (dropout): Dropout(p=0.1, inplace=False)
      )
    )
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (decoder): TransformerDecoder(
    (embedding): Embedding(11050, 256, padding_id

In [ ]:
def translate_sentence_transformer(
    sentence,
    model,
    english_vocab,
    french_vocab,
    device,
    max_len=50
):

    model.eval()

    # Tokenize English sentence
    tokens = tokenize(sentence)

    # Convert English words to IDs
    src_indexes = [
        english_vocab.word2idx.get(
            token,
            english_vocab.word2idx["<UNK>"]
        )
        for token in tokens
    ]

    # Add SOS and EOS
    src_indexes = (
        [english_vocab.word2idx["<SOS>"]]
        + src_indexes
        + [english_vocab.word2idx["<EOS>"]]
    )

    # Convert to tensor
    src_tensor = torch.tensor(
        src_indexes,
        dtype=torch.long,
        device=device
    ).unsqueeze(0)

    # Source padding mask
    src_padding_mask = create_src_mask(src_tensor)

    # Encode
    with torch.no_grad():

        encoder_output, _ = model.encoder(
            src_tensor,
            src_padding_mask
        )

    # Start decoder with SOS
    trg_indexes = [
        french_vocab.word2idx["<SOS>"]
    ]

    for _ in range(max_len):

        trg_tensor = torch.tensor(
            trg_indexes,
            dtype=torch.long,
            device=device
        ).unsqueeze(0)

        # Create causal mask
        trg_mask = create_trg_mask(trg_tensor)

        # Target padding mask
        trg_padding_mask = create_trg_padding_mask(trg_tensor)

        with torch.no_grad():

            output, _, _ = model.decoder(
                trg_tensor,
                encoder_output,
                trg_mask,
                trg_padding_mask,
                src_padding_mask
            )

        # Prediction for the last token
        pred_token = output[:, -1, :].argmax(1).item()

        # Add predicted token
        trg_indexes.append(pred_token)

        # Stop if EOS is generated
        if pred_token == french_vocab.word2idx["<EOS>"]:
            break

    # Convert IDs back to French words
    trg_tokens = [
        french_vocab.idx2word[i]
        for i in trg_indexes
    ]

    return trg_tokens

In [ ]:
sentence = val_df.iloc[0]["English words/sentences"]

translation = translate_sentence_transformer(
    sentence,
    model,
    english_vocab,
    french_vocab,
    device
)

print("English:")
print(sentence)

print("\nPredicted French:")
print(" ".join(translation))

English:
Why didn't you follow my advice?

Predicted French:
<SOS> pourquoi vous ne vous m'avez pas mon conseil ? <EOS>


In [ ]:
print(
    " ".join(
        translation[1:-1]
    )
)

pourquoi vous ne vous m'avez pas mon conseil ?


In [ ]:
from nltk.translate.bleu_score import corpus_bleu
def evaluate_bleu_transformer(
    model,
    df,
    english_vocab,
    french_vocab,
    device,
    n_samples=500
):

    references = []
    hypotheses = []

    for i in range(min(n_samples, len(df))):

        english_sentence = df.iloc[i]["English words/sentences"]
        french_sentence = df.iloc[i]["French words/sentences"]

        # Generate translation
        prediction = translate_sentence_transformer(
            english_sentence,
            model,
            english_vocab,
            french_vocab,
            device
        )

        # Remove special tokens
        prediction = [
            word for word in prediction
            if word not in ["<SOS>", "<EOS>", "<PAD>"]
        ]

        reference = tokenize(french_sentence)

        references.append([reference])
        hypotheses.append(prediction)

    bleu = corpus_bleu(
        references,
        hypotheses
    )

    return bleu


In [ ]:
bleu_transformer = evaluate_bleu_transformer(
    model,
    val_df,
    english_vocab,
    french_vocab,
    device,
    n_samples=500
)

print("Transformer BLEU:", bleu_transformer)
print("Transformer BLEU × 100:", bleu_transformer * 100)

Transformer BLEU: 0.16061424774432836
Transformer BLEU × 100: 16.061424774432837
